In [1]:
pip install azure-ai-projects azure-core azure-storage-blob

In [2]:
pip install --upgrade openai azure-ai-projects azure-identity

In [11]:
from openai import OpenAI
import os
from google.colab import userdata

# Retrieve the secret key from Google Colab
api_key = userdata.get("New_Open_AI")

client = OpenAI(
    api_key=api_key,  # Picks up the secret key from Colab environment
    base_url="https://21-april.openai.azure.com/openai/v1"
)

In [12]:
# 1. Define the input data
# In a real scenario, you can load this from a .txt or .json file
resume_data = """
RICHARD SANCHEZ
Education: Master of Business Management (2029-2030)
Work: Marketing Manager at Borcelle Studio (2030-Present)
Skills: Project Management, PR, Leadership...
"""

# 2. Use the Responses API for structured extraction
# Use the structured input pattern for GPT-4.1
# Updated call with 2026-compliant type naming
try:
    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",  # Changed from 'text'
                        "text": "Extract the following details into a JSON object: Name, Education, Latest Job, and Top 3 Skills. Return ONLY valid JSON."
                    },
                    {
                        "type": "input_text",  # Changed from 'text'
                        "text": f"RESUME SOURCE:\n{resume_data}"
                    }
                ]
            }
        ]
    )

    print("--- EXTRACTED DATA ---")
    print(response.output[0].content)

except Exception as e:
    print(f"Extraction Error: {e}")

--- EXTRACTED DATA ---
[ResponseOutputText(annotations=[], text='{\n  "Name": "Richard Sanchez",\n  "Education": "Master of Business Management (2029-2030)",\n  "Latest Job": "Marketing Manager at Borcelle Studio (2030-Present)",\n  "Top 3 Skills": ["Project Management", "PR", "Leadership"]\n}', type='output_text', logprobs=[])]


In [13]:
import json

try:
    # 1. Get the first item in the content list
    # 2. Access the 'text' attribute of that item
    output_text_item = response.output[0].content[0]

    # In some SDK versions, it's a dict; in others, it's an object.
    # This handles both:
    if isinstance(output_text_item, dict):
        raw_json_string = output_text_item.get("text", "")
    else:
        raw_json_string = output_text_item.text

    # Now parse the actual string
    extracted_data = json.loads(raw_json_string)

    print("--- SUCCESS: DATA PARSED ---")
    print(json.dumps(extracted_data, indent=2))

except Exception as e:
    print(f"Parsing Error: {e}")
    # Troubleshooting: print the type to see what exactly was returned
    print(f"Actual content type: {type(response.output[0].content[0])}")

--- SUCCESS: DATA PARSED ---
{
  "Name": "Richard Sanchez",
  "Education": "Master of Business Management (2029-2030)",
  "Latest Job": "Marketing Manager at Borcelle Studio (2030-Present)",
  "Top 3 Skills": [
    "Project Management",
    "PR",
    "Leadership"
  ]
}


In [20]:

import os
from azure.storage.blob import BlobServiceClient
from google.colab import userdata
key=userdata.get('AZURE_STORAGE_CONNECTION_STRING_1')

conn_str = key

blob_service_client = BlobServiceClient.from_connection_string(conn_str)

container_name = "21-april"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob="result.txt"
)

blob_client.upload_blob("Test upload", overwrite=True)

print("✅ Upload successful")

✅ Upload successful


In [8]:
import datetime

file_name = f"resume_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

In [9]:
blob_client = blob_service_client.get_blob_client(
    container="21-april",
    blob=file_name
)

blob_client.upload_blob(response.output_text, overwrite=True)

{'etag': '"0x8DE9F6D6DE2FB06"',
 'last_modified': datetime.datetime(2026, 4, 21, 6, 15, 47, tzinfo=datetime.timezone.utc),
 'content_md5': bytearray(b'\xdb\xe3g\xe1$0\xd2G\xd7k\xff\x97\xbd\xe3\xdb`'),
 'client_request_id': '89a3956e-3d49-11f1-8061-0242ac1c000c',
 'request_id': 'da4c2550-b01e-0014-7656-d14cf0000000',
 'version': '2026-02-06',
 'version_id': None,
 'date': datetime.datetime(2026, 4, 21, 6, 15, 47, tzinfo=datetime.timezone.utc),
 'request_server_encrypted': True,
 'encryption_key_sha256': None,
 'encryption_scope': None,
 'structured_body': None}

In [10]:
print(f"✅ Stored as: {file_name}")

✅ Stored as: resume_20260421_061508.txt


In [ ]:
# Build & Deploy an AI Research Paper Summarizer (Azure AI Foundry)
# 🎯 Objective
# Students will:

# Create AI resource

# Build an AI agent

# Deploy it

# Call it using Python (Colab)

# Store summarized output in cloud

<!-- Build & Deploy an AI Research Paper Summarizer (Azure AI Foundry)
🎯 Objective
Students will:

Create AI resource

Build an AI agent

Deploy it

Call it using Python (Colab)

Store summarized output in cloud -->

In [11]:
from openai import OpenAI
import os
from google.colab import userdata

# Retrieve the secret key from Google Colab
api_key = userdata.get("New_Open_AI_2")

client = OpenAI(
    api_key=api_key,  # Picks up the secret key from Colab environment
    base_url="https://21april2.openai.azure.com/openai/v1"
)

In [12]:
research_data ='''Title: Attention Is All You Need

Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin

Abstract:
The dominant sequence transduction models are based on complex recurrent or convolutional neural networks. In this paper, we propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show that these models are superior in quality while being more parallelizable and requiring significantly less time to train.

Introduction:
Sequence transduction tasks such as machine translation, speech recognition, and text summarization have traditionally relied on recurrent neural networks (RNNs) and convolutional neural networks (CNNs). However, these approaches suffer from limitations in parallelization and long-range dependency modeling. The Transformer model addresses these challenges using a self-attention mechanism.

Methodology:
The Transformer architecture is based on an encoder-decoder structure. Each encoder layer consists of a self-attention mechanism followed by a feed-forward neural network. The decoder includes an additional attention layer that attends to the encoder output. Positional encoding is added to retain sequence order information. Multi-head attention allows the model to focus on different parts of the sequence simultaneously.

Key Contributions:

1. Introduction of the Transformer architecture without recurrence or convolution.
2. Use of self-attention as the primary mechanism for sequence modeling.
3. Significant improvement in training efficiency and parallelization.
4. State-of-the-art performance on machine translation benchmarks.

Results:
The Transformer achieves better BLEU scores compared to previous models on benchmark datasets such as WMT 2014 English-to-German and English-to-French tasks. It also requires less training time due to parallel computation.

Conclusion:
The Transformer model demonstrates that attention mechanisms alone are sufficient for high-quality sequence transduction tasks. It opens the door for more efficient and scalable models in natural language processing.

Limitations:
The model requires large amounts of data and computational resources. It may also struggle with extremely long sequences without additional optimization techniques.'''


try:
    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": """
Extract the following details from the research paper and return ONLY valid JSON:

{
  "title": "",
  "authors": [],
  "abstract_summary": "",
  "methodology": "",
  "key_contributions": [],
  "conclusion": "",
  "limitations": ""
}
"""
                    },
                    {
                        "type": "input_text",
                        "text": f"PAPER TEXT:\n{research_data}"
                    }
                ]
            }
        ]
    )

    print("--- EXTRACTED DATA ---")
    print(response.output[0].content)

except Exception as e:
    print(f"Extraction Error: {e}")






--- EXTRACTED DATA ---
[ResponseOutputText(annotations=[], text='```json\n{\n  "title": "Attention Is All You Need",\n  "authors": [\n    "Ashish Vaswani",\n    "Noam Shazeer",\n    "Niki Parmar",\n    "Jakob Uszkoreit",\n    "Llion Jones",\n    "Aidan N. Gomez",\n    "Lukasz Kaiser",\n    "Illia Polosukhin"\n  ],\n  "abstract_summary": "The paper introduces the Transformer, a new sequence transduction model architecture based solely on attention mechanisms, removing the need for recurrence or convolutions. The Transformer demonstrates superior quality, greater parallelizability, and reduced training time in machine translation tasks compared to previous models.",\n  "methodology": "The Transformer uses an encoder-decoder structure where each encoder layer has a self-attention mechanism and a feed-forward neural network. The decoder layers contain an additional encoder-decoder attention mechanism. Positional encoding is added to inject sequence order information. Multi-head attention i

In [14]:
import json

try:
    raw_json_string = response.output_text

    extracted_data = json.loads(raw_json_string)

    print("--- SUCCESS: DATA PARSED ---")
    print(json.dumps(extracted_data, indent=2))

except Exception as e:
    print(f"Parsing Error: {e}")
    print("Raw output was:\n", response.output_text)

Parsing Error: Expecting value: line 1 column 1 (char 0)
Raw output was:
 ```json
{
  "title": "Attention Is All You Need",
  "authors": [
    "Ashish Vaswani",
    "Noam Shazeer",
    "Niki Parmar",
    "Jakob Uszkoreit",
    "Llion Jones",
    "Aidan N. Gomez",
    "Lukasz Kaiser",
    "Illia Polosukhin"
  ],
  "abstract_summary": "The paper introduces the Transformer, a new sequence transduction model architecture based solely on attention mechanisms, removing the need for recurrence or convolutions. The Transformer demonstrates superior quality, greater parallelizability, and reduced training time in machine translation tasks compared to previous models.",
  "methodology": "The Transformer uses an encoder-decoder structure where each encoder layer has a self-attention mechanism and a feed-forward neural network. The decoder layers contain an additional encoder-decoder attention mechanism. Positional encoding is added to inject sequence order information. Multi-head attention is emp

In [21]:

import os
from azure.storage.blob import BlobServiceClient
from google.colab import userdata
key=userdata.get('AZURE_STORAGE_CONNECTION_STRING_2')

conn_str = key

blob_service_client = BlobServiceClient.from_connection_string(conn_str)

container_name = "21-april-2"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob="result.txt"
)

blob_client.upload_blob("Test upload", overwrite=True)

print("✅ Upload successful")

✅ Upload successful


In [20]:
import datetime

file_name = f"research_paper{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

In [21]:
blob_client = blob_service_client.get_blob_client(
    container="21-april-2",
    blob=file_name
)

blob_client.upload_blob(response.output_text, overwrite=True)

{'etag': '"0x8DE9F71D9CA572C"',
 'last_modified': datetime.datetime(2026, 4, 21, 6, 47, 26, tzinfo=datetime.timezone.utc),
 'content_md5': bytearray(b'\xcfqZ \x96\x94\x8c\xda%]2\x90d=\x05\x85'),
 'client_request_id': 'f5896dcc-3d4d-11f1-8061-0242ac1c000c',
 'request_id': 'c7f4db15-f01e-005c-175a-d17e6d000000',
 'version': '2026-02-06',
 'version_id': None,
 'date': datetime.datetime(2026, 4, 21, 6, 47, 26, tzinfo=datetime.timezone.utc),
 'request_server_encrypted': True,
 'encryption_key_sha256': None,
 'encryption_scope': None,
 'structured_body': None}

In [22]:
print(f"✅ Stored as: {file_name}")

✅ Stored as: research_paper20260421_064721.txt


In [ ]:
# Build & Deploy an AI Medical Report Interpreter (Azure AI Foundry)
# 🎯 Objective
# Students will:

# Create AI resource

# Build an AI agent

# Deploy it

# Call it using Python (Colab)

# Store interpreted reports in cloud

In [23]:
from openai import OpenAI
import os
from google.colab import userdata

# Retrieve the secret key from Google Colab
api_key = userdata.get("New_Open_AI_3")

client = OpenAI(
    api_key=api_key,  # Picks up the secret key from Colab environment
    base_url="https://21-april-3.openai.azure.com/openai/v1"
)

In [25]:
medical_data='''Patient Name: Sarah Williams
Age: 52
Gender: Female

Test Type: Comprehensive Blood Panel

Hemoglobin: 9.5 g/dL (Low)
WBC Count: 13500 cells/mcL (High)
Platelet Count: 250000 cells/mcL (Normal)
Fasting Blood Glucose: 165 mg/dL (High)
Cholesterol: 240 mg/dL (High)
HDL: 38 mg/dL (Low)
LDL: 160 mg/dL (High)

Liver Function Test:
ALT: 55 U/L (High)
AST: 48 U/L (High)

Kidney Function Test:
Creatinine: 1.3 mg/dL (Slightly High)
Urea: 45 mg/dL (High)

Symptoms:

* Fatigue
* Frequent urination
* Mild chest discomfort

Doctor Notes:
The patient shows elevated glucose and cholesterol levels along with mild liver enzyme elevation. Further evaluation is recommended.
'''
try:
    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": """
Extract the following details from the medical report and return ONLY valid JSON.

Do NOT include explanations or extra text.

{
  "patient_name": "",
  "age": "",
  "gender": "",
  "test_type": "",
  "key_findings": [],
  "abnormal_values": [],
  "summary": "",
  "risk_level": "Low | Moderate | High"
}
"""
                    },
                    {
                        "type": "input_text",
                        "text": f"MEDICAL REPORT:\n{medical_data}"
                    }
                ]
            }
        ]
    )

    print("--- EXTRACTED DATA ---")
    print(response.output_text)   # ✅ updated (important)

except Exception as e:
    print(f"Extraction Error: {e}")

--- EXTRACTED DATA ---
{
  "patient_name": "Sarah Williams",
  "age": "52",
  "gender": "Female",
  "test_type": "Comprehensive Blood Panel",
  "key_findings": [
    "Low hemoglobin",
    "High WBC count",
    "High fasting blood glucose",
    "High cholesterol",
    "Low HDL",
    "High LDL",
    "High ALT",
    "High AST",
    "Slightly high creatinine",
    "High urea",
    "Fatigue",
    "Frequent urination",
    "Mild chest discomfort"
  ],
  "abnormal_values": [
    "Hemoglobin: 9.5 g/dL (Low)",
    "WBC Count: 13500 cells/mcL (High)",
    "Fasting Blood Glucose: 165 mg/dL (High)",
    "Cholesterol: 240 mg/dL (High)",
    "HDL: 38 mg/dL (Low)",
    "LDL: 160 mg/dL (High)",
    "ALT: 55 U/L (High)",
    "AST: 48 U/L (High)",
    "Creatinine: 1.3 mg/dL (Slightly High)",
    "Urea: 45 mg/dL (High)"
  ],
  "summary": "The patient has anemia, leukocytosis, hyperglycemia, hypercholesterolemia, dyslipidemia, mild elevation of liver enzymes, and mildly impaired kidney function. She also 

In [22]:


import os
from azure.storage.blob import BlobServiceClient
from google.colab import userdata
key=userdata.get('AZURE_STORAGE_CONNECTION_STRING_3')

conn_str = key

blob_service_client = BlobServiceClient.from_connection_string(conn_str)

container_name = "21-april-3"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob="result.txt"
)

blob_client.upload_blob("Test upload", overwrite=True)

print("✅ Upload successful")

✅ Upload successful


In [30]:
import datetime

file_name = f"medical_report{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

In [31]:
blob_client = blob_service_client.get_blob_client(
    container="21-april-3",
    blob=file_name
)

blob_client.upload_blob(response.output_text, overwrite=True)

{'etag': '"0x8DE9F7536331890"',
 'last_modified': datetime.datetime(2026, 4, 21, 7, 11, 30, tzinfo=datetime.timezone.utc),
 'content_md5': bytearray(b'\xadE\x94\x0f\xae\x96fX\x13\x93g\xb45t\xc3\xb5'),
 'client_request_id': '51f3d450-3d51-11f1-8061-0242ac1c000c',
 'request_id': '68b29778-001e-00a9-7f5e-d14a15000000',
 'version': '2026-02-06',
 'version_id': None,
 'date': datetime.datetime(2026, 4, 21, 7, 11, 29, tzinfo=datetime.timezone.utc),
 'request_server_encrypted': True,
 'encryption_key_sha256': None,
 'encryption_scope': None,
 'structured_body': None}

In [32]:
print(f"✅ Stored as: {file_name}")

✅ Stored as: medical_report20260421_071105.txt
